In [ ]:
import torch
import torch.nn as nn

import numpy as np
import matplotlib.pyplot as plt
import torch.optim as optim

In [ ]:
class MyModel(nn.Module):
    def __init__(self, input, output, hidden_size=2048, choice='tanh'):
        super().__init__()
        self.activations = nn.ModuleDict({
            'tanh': nn.Tanh(),
            'sigm': nn.Sigmoid()
        })
        self.layers = nn.ModuleList()
        for i in range(10):
            self.layers.add_module(f'layer_{i}', nn.Linear(input, hidden_size))
            self.layers.add_module(f'act_{i}', self.activations[choice])
            input = hidden_size
            hidden_size = int(hidden_size/2)
        self.layers.add_module('foutput_{i}', nn.Linear(input, output))

    def forward(self, x):
        output = []
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i != 0 and i%2 == 0 and i%4 != 0:
                output.append(x)
                
        output.append(x)
        return output

In [ ]:
class PINN(nn.Module):
    
    def __init__(self,
                 layers: list):
        super().__init__()
        self.layers = nn.ModuleList()
        for i in range(len(layers) - 1):
            if i == 0:
                self.layers.append(SinLayer(layers[i],
                                            layers[i+1], is_first=True))
            elif i == len(layers) - 2:
                self.layers.append(nn.Linear(layers[i],
                                             layers[i+1],))
            else:
                self.layers.append(SinLayer(layers[i],
                                            layers[i+1], is_first=False))

    def forward(self,
                x: torch.Tensor,
                y: torch.Tensor,
                t: torch.Tensor) -> torch.Tensor:

        inputs = torch.cat([x.reshape(-1, 1),
                            y.reshape(-1, 1),
                            t.reshape(-1, 1)], dim=1)
        for layer in self.layers:
            inputs = layer(inputs)
        return inputs

In [ ]:
epochs = 1000

loss_history = {'pde': [],
                'ic': [],
                'bc': []}

for epoch in range(epochs+1):
    optimizer.zero_grad()

    u_pred_i = model(x_i, y_i, t_i)
    loss_ic = mse_loss(u_pred_i, u_i)  # потери на начальных условиях (ic - initial conditions)

    u_pred_b = model(x_b, y_b, t_b)
    loss_bc = mse_loss(u_pred_b, u_b)  # потери на граничных условиях (bc - boundary conditions)

    loss_pde = mse_loss(pde_residual(model, x_f, y_f, t_f, ALPHA),
                        torch.zeros_like(x_f))  # потери модели по реализованной функции

    loss = w_ic * loss_ic + w_bc * loss_bc + w_pde * loss_pde

    loss.backward()
    optimizer.step()

    loss_history['ic'].append(loss_ic.item())
    loss_history['bc'].append(loss_bc.item())
    loss_history['pde'].append(loss_pde.item())

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.3e}, "
              f"IC: {loss_ic.item():.3e}, "
              f"BC: {loss_bc.item():.3e}, "
              f"PDE: {loss_pde.item():.3e}")